# Modeling

Packages and setup

In [ ]:
# Imports & settings
from foodcast.imports import *
notebook_settings() 
os.chdir(PROJECT_ROOT)
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_4, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, DATA_DIR_3_4, DATA_DIR_3_5, DATA_DIR_3_6, DATA_DIR_3_7, DATA_DIR_3_8, _ = DATA_DIR_3_x
from foodcast.tools.rolling import rolling_window_avg, add_interday_variables, add_intraday_variables, unroll, season_from_month
from foodcast.tools.takeout import takeout
from foodcast.tools.labeling_functions import rename_items_by_modifications, rename_items
takeout = takeout + '|take and|take n\''

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
data = load_all_res_3_4_ai()

# Load special data
grouping_mappings = pd.read_pickle(DATA_DIR_3 / 'grouping_mappings.pkl')

totals = [0, 0, 0, 0]
for loc_id in location_ids_by_coverage:
    totals[0] += data[loc_id].shape[0]

data_c = {}
for loc_id in tqdm(location_ids_by_coverage):
    
    df = (
        data[loc_id]
        .query('~is_drink')
        .query('~is_nonfood')
        .query('~is_nonmeal_merchandise'))

    totals[1] += df.shape[0]
    filepath = Path(DATA_DIR_3_5) / f'{loc_id}.parquet'
    #if not filepath.exists():
    df.to_parquet(filepath, index=False)

    df = (
        df
        .fillna({'item_modifications':''})
        .query('~item_modifications.str.lower().str.contains(@takeout)')
        .query('~item_name.str.lower().str.contains(@takeout)'))
    
    totals[2] += df.shape[0]
    filepath = Path(DATA_DIR_3_6) / f'{loc_id}.parquet'
    #if not filepath.exists():
    df.to_parquet(filepath, index=False)
    
    if loc_id in ['EMBVNVD207CC6','C0BE4NDSW26QN','75WYSXR9QBK5M','V3Q26BHF3SE2H','LBZEEFSBJNB3Z',
                  'SAFK7ND1HR6XS','CB2KHY1C2G9PT','S8MT0YGD2KTN9','LFZFT3VASXPED','1SQPTEGYPH0GA',
                  '9XKJD8DQTH559','LQ5EH4BKGV61T','78AY09MVJVTYE']:
        
        mapping = grouping_mappings[loc_id]
        df = (
            df
            .assign(item_name = lambda df: df.item_name.replace(mapping)))
    
        items_less_than_2_dollars = (
            df
            .groupby('item_name')
            ['unit_price']
            .max()
            .to_frame(name='max_unit_price')
            .query('max_unit_price < 200.0')
            .reset_index()
            .item_name
            .tolist())
        
        items_less_than_10_sales = df.item_name.value_counts().to_frame('c').query('c < 10').index.tolist()
    
        df = (
            df
            .query('~item_name.isin(@items_less_than_2_dollars)')
            .query('~item_name.isin(@items_less_than_10_sales)')
            .query('~item_name.isin(["to_remove","catering_takeout"])')
            )
    
        totals[3] += df.shape[0]
        filepath = Path(DATA_DIR_3_7) / f'{loc_id}.parquet'
        #if not filepath.exists():
        df.to_parquet(filepath, index=False)
    
    data_c[loc_id] = df
        
print(totals)

In [ ]:
for loc_id in location_ids_by_coverage:
    
    if loc_id in ["EMBVNVD207CC6","C0BE4NDSW26QN","75WYSXR9QBK5M","V3Q26BHF3SE2H","LBZEEFSBJNB3Z",
                  "SAFK7ND1HR6XS","CB2KHY1C2G9PT","S8MT0YGD2KTN9","LFZFT3VASXPED","1SQPTEGYPH0GA",
                  "9XKJD8DQTH559","LQ5EH4BKGV61T","78AY09MVJVTYE"]:
        
        df = data_c[loc_id]
        print(loc_id)
        
        presence_dict = {}
        for item, group in df.groupby("item_name"):
            presence_dict[item] = infer_active_days(group["created_at"], max_gap_days=120)
        presence_df = pd.concat(presence_dict, axis=1).fillna(False)

        presence_weekly = presence_df.resample('W').max().fillna(False).astype(bool).to_period('W')
        dish_order = list(df.item_name.value_counts().index)
        plot_boolean_time_series(presence_weekly, loc_id, before_after_details_true, dish_order, [])
        
        plot_dish_time_series(df.set_index('created_at'), loc_id, before_after_details_true, top_n=70)

        presence_daily = strict_bridge_fill(presence_df, limit=7).resample('D').max()
        #presence_daily = (presence_weekly.astype(float).replace(0.0, np.nan)[vegetarian_dishes].resample('D').interpolate(limit=6).fillna(0).astype(int).sum(axis=1).plot())
        menu = pd.read_csv(Path("scripts") / "labeling" / "dish_labels_t2" / (loc_id + "_1.csv"))

        vegan_dishes = menu.loc[menu["vegan"], "item_name"] 
        vegetarian_dishes = menu.loc[menu["vegetarian"], "item_name"] 
        mpbamod_dishes = menu.loc[menu["mpbamod"], "item_name"] 
        lamb_dishes = menu.loc[menu["lamb"], "item_name"] 
        chunked_beef_or_pork_dishes = menu.loc[menu["chunked_beef_or_pork"], "item_name"] 
        pulled_pork_dishes = menu.loc[menu["pulled_pork"], "item_name"] 
        beef_or_pork_burger_dishes = menu.loc[menu["beef_or_pork_burger"], "item_name"] 
        ground_meat_dishes = menu.loc[menu["ground_meat"], "item_name"] 
        meatballs_dishes = menu.loc[menu["meatballs"], "item_name"] 
        sausage_dishes = menu.loc[menu["sausage"], "item_name"] 
        bacon_dishes = menu.loc[menu["bacon"], "item_name"] 
        breakfast_sausage_patty_dishes = menu.loc[menu["breakfast_sausage_patty"], "item_name"] 
        unfried_chicken_dishes = menu.loc[menu["unfried_chicken"], "item_name"] 
        fried_chicken_dishes = menu.loc[menu["fried_chicken"], "item_name"] 
        savory_dairy_dishes = menu.loc[menu["savory_dairy"], "item_name"] 
        sweet_dairy_dishes = menu.loc[menu["sweet_dairy"], "item_name"] 
        egg_dishes = menu.loc[menu["egg"], "item_name"]

        category_lists = {
            "vegan_dishes_count": vegan_dishes,
            "vegetarian_dishes_count": vegetarian_dishes,
            "mpbamod_dishes_count": mpbamod_dishes,
            "lamb_dishes_count": lamb_dishes,
            "chunked_beef_or_pork_dishes_count": chunked_beef_or_pork_dishes,
            "pulled_pork_dishes_count": pulled_pork_dishes,
            "beef_or_pork_burger_dishes_count": beef_or_pork_burger_dishes,
            "ground_meat_dishes_count": ground_meat_dishes,
            "meatballs_dishes_count": meatballs_dishes,
            "sausage_dishes_count": sausage_dishes,
            "bacon_dishes_count": bacon_dishes,
            "breakfast_sausage_patty_dishes_count": breakfast_sausage_patty_dishes,
            "unfried_chicken_dishes_count": unfried_chicken_dishes,
            "fried_chicken_dishes_count": fried_chicken_dishes,
            "savory_dairy_dishes_count": savory_dairy_dishes,
            "sweet_dairy_dishes_count": sweet_dairy_dishes,
            "egg_dishes_count": egg_dishes,
        }

        # Total dishes available that day
        dishes_count = presence_daily.sum(axis=1)

        # Compute each category safely
        counts = {}
        for name, dish_list in category_lists.items():
            cols = presence_daily.columns.intersection(dish_list)
            counts[name] = presence_daily[cols].sum(axis=1)

        # Assemble final dataframe
        menu_counts = pd.concat(
            [dishes_count] + [counts[k] for k in category_lists.keys()],
            axis=1
        )

        menu_counts.columns = [
            "dishes_count",
            *category_lists.keys(),
        ]

        menu_counts.plot()
        menu_counts.to_csv(Path("scripts") / "labeling" / "dish_counts" / f"{loc_id}.csv")